# LatentSpace: training pipeline on Colab (T4 or L4)

Same stages as `notebooks/kaggle_train.ipynb` (the main notebook), on one GPU, so the two
GRPO runs and the evaluations run one after another. Runtime → Change runtime type → **T4**
(fp16) or **L4** (bf16, faster; `precision: auto` picks it). Secrets (key icon): add
**`HF_TOKEN`** with notebook access.

Colab Free sessions stop after a few hours: run with `PUSH = True` and resume from the Hub.

## Time and cost estimate (Kaggle numbers; one T4 doubles stages 4-5 wall time, an L4 is ~1.5-2x faster)

| Stage | Where | Kaggle 2x T4 (wall) | GPU-h | HF credits |
|---|---|---|---|---|
| 0 install + tests | CPU | 8-12 min | - | - |
| 1 baseline, local fp16 | GPU 0 | ~25-35 min / 100 tasks | 0.5 | - |
| 1 baseline, router (optional) | API | ~5-10 min / 100 tasks (parallel to the local one) | - | ~$0.05-0.10 |
| 2 teacher_collect, aya-expanse-32b router | API | ~1.5-2 h / 600 trajectories (sequential calls) | - | ~$1.5-2.5, capped by `MAX_USD` |
| 2' teacher_collect, 8B 4-bit local fallback | GPU 0 | ~4-5 h / 300 trajectories | 4-5 | - |
| 3 distill, QLoRA 4-bit fp16 | GPU 0 | ~30-60 min / 400 rows x 2 epochs | 0.5-1 | - |
| 4 grpo run_A ‖ run_B, QLoRA, 100 steps | GPU 0 ‖ GPU 1 | ~3-3.5 h (both at once) | 6-7 | - |
| 5 evaluate, 4 checkpoints x 100 tasks x 5 trials, fp16 | GPU 0 ‖ GPU 1 | ~4-5 h | 8-10 | - |

Assumptions: Tiny Aya 3.35B; HF `generate` at ~20 tok/s (fp16) / ~10 tok/s (4-bit) per
sequence on a T4; ~300 generated tokens and ~2.5 turns per rollout; AgentTune rolls out one
trajectory at a time. Teacher cost at $0.50 / $1.50 per M input / output tokens (Cohere's list
price for Aya Expanse 32B; check the provider price on the model page). Scale linearly with the
number of tasks. Kaggle sessions stop at 12 h, so this notebook is meant to be run as three
sessions (stages 0-3, stage 4, stage 5); with `PUSH = True` each session picks up the previous
one's adapters and data from the Hub.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
try:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
import os, subprocess
BRANCH = "training-pipeline"   # switch to "main" once merged
REPO = "/content/LatentSpace"
gh = os.environ.get("GITHUB_TOKEN")
url = f"https://{gh}@github.com/kushwaha001/LatentSpace" if gh else "https://github.com/kushwaha001/LatentSpace"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
subprocess.run(["bash", f"{REPO}/scripts/install_deps.sh", "/content/AgentTune"], check=True)

In [ ]:
import os, subprocess, sys, json, time, glob
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/src"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def stage(*args, gpu=None, log=None):
    """Run one pipeline stage; stream its output. gpu=0/1 pins it to one GPU."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    cmd = [sys.executable, "-m", "latentspace_train", *args]
    print("$", " ".join(cmd), f"(GPU {gpu})" if gpu is not None else "")
    if log:  # background: returns the process, output goes to the log file
        return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    p = subprocess.run(cmd, env=env)
    if p.returncode:
        raise RuntimeError(f"stage failed: {' '.join(args)}")

def wait_all(procs, logs, every=300):
    """Wait for background stages, printing the last log line every `every` seconds."""
    while any(p.poll() is None for p in procs):
        time.sleep(every)
        for l in logs:
            tail = open(l).read().strip().splitlines()[-1:] if os.path.exists(l) else []
            print(time.strftime("%H:%M"), l, "|", tail[0][-160:] if tail else "")
    for p, l in zip(procs, logs):
        print(l, "exit", p.returncode)
        if p.returncode:
            print(open(l).read()[-3000:])
    assert all(p.returncode == 0 for p in procs), "a background stage failed (log above)"

def show(path):
    d = json.load(open(path))
    print(path, "| seconds", d.get("seconds"), "| commit", d["provenance"]["commit"][:8])
    return d

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
STUDENT     = "tiny-aya-global"   # or tiny-aya-fire / tiny-aya-water / tiny-aya-earth
PUSH        = True                # push adapters, logs, datasets to the Hub (needs a write token)
HF_USER     = "kanishk001"        # Hub namespace for every repo the pipeline creates
MAX_USD     = 5.0                 # teacher_collect budget (HF credits, Inference Providers)
API_BASELINE = True               # also run the cheap router baseline of tiny-aya-global
LIMIT       = None                # e.g. 20 for a quick end-to-end check on real data
TEACHER_BACKEND = "router"        # "router" (aya-expanse-32b, HF credits) or "local" (8B 4-bit on GPU)

COMMON = [f"--set=student.model={STUDENT}", f"--set=hub.namespace={HF_USER}"]
if LIMIT:
    COMMON.append(f"--limit={LIMIT}")
PUSH_FLAG = ["--push"] if PUSH else []
HUB = lambda name: f"{HF_USER}/{name}"

In [ ]:
# The transit tools (src/transit_agent/) and eval set (data/eval/) come from the eval-set
# branch. Merge it if this checkout does not have them yet; otherwise fall back to the stub.
if not (os.path.isdir(f"{REPO}/src/transit_agent") and os.path.isdir(f"{REPO}/data/eval")):
    r = subprocess.run(f"cd {REPO} && git fetch -q origin eval-set && "
                       "git -c user.name=nb -c user.email=nb@local merge -q --no-edit origin/eval-set",
                       shell=True)
    print("merged eval-set" if r.returncode == 0 else "eval-set branch not available")
if os.path.isdir(f"{REPO}/src/transit_agent") and os.path.isdir(f"{REPO}/data/eval"):
    print("transit tools + eval set present:", sorted(os.listdir(f"{REPO}/data/eval")))
else:
    print("WARNING: no transit tools / eval set -> using the stub tasks (wiring check only)")
    COMMON.append("--set=tasks.source=stub")

In [ ]:
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)
stage("grpo", "--run", "run_B", "--dry-run")

In [ ]:
stage("baseline", *COMMON, *PUSH_FLAG)
if API_BASELINE:
    stage("baseline", *COMMON, *PUSH_FLAG, "--set=baseline.modes=[api]",
          "--set=results_dir=results/api_baseline")
show("results/10_baseline.json")["modes"]["local"]["summary"]["overall"]

In [ ]:
stage("teacher_collect", *COMMON, *PUSH_FLAG, f"--max-usd={MAX_USD}", f"--set=teacher.backend={TEACHER_BACKEND}")
t = show("results/11_teacher_collect.json"); print("kept", t["kept"], "of", t["seen"], t["budget"])

In [ ]:
stage("distill", *COMMON, *PUSH_FLAG)

In [ ]:
RESUME_FROM_HUB = not os.path.isdir(f"{REPO}/outputs/distill/adapter")
START = [f"--set=student.adapter={HUB('tiny-aya-global-transit-distill-lora')}"] if RESUME_FROM_HUB else []
for r in ("run_A", "run_B"):   # one GPU: sequential, identical settings
    stage("grpo", "--run", r, *COMMON, *START, *PUSH_FLAG)
    d = show(f"results/13_grpo_{r}.json"); print(r, d["global_step"], d["error"], d["shared_settings_fingerprint"])

In [ ]:
def ck(name, repo):
    local = f"{REPO}/outputs/{'distill' if name == 'distill' else 'grpo/' + name}/adapter"
    return local if os.path.isdir(local) else HUB(repo)
ADAPTERS = {"distill": ck("distill", "tiny-aya-global-transit-distill-lora"),
            "run_A": ck("run_A", "tiny-aya-global-transit-grpo-default-reward-lora"),
            "run_B": ck("run_B", "tiny-aya-global-transit-grpo-audit-reward-lora")}
CKPTS = "[{name: base}, " + ", ".join(f"{{name: {n}, adapter: '{a}'}}" for n, a in ADAPTERS.items()) + "]"
stage("evaluate", *COMMON, *PUSH_FLAG, f"--set=evaluate.checkpoints={CKPTS}")

In [ ]:
e = show("results/14_evaluate.json")
rows = []
for name, c in e["checkpoints"].items():
    if "summary" not in c:
        print(name, c); continue
    o = c["summary"]["overall"]; rel = o.get("reliability", {})
    rows.append((name, o["success"], rel.get("pass_hat_k"), o["tool_call_validity"], o["audit_combined"],
                 o["agent_loop_detection"], o["unconfirmed_irreversible_rate"]))
print(f"{'ckpt':8} {'success':>8} {'pass^5':>7} {'valid':>6} {'reward':>7} {'loopfree':>8} {'unconf':>6}")
for r in rows:
    print(f"{r[0]:8} " + " ".join(f"{x:>7.3f}" if isinstance(x, float) else f"{str(x):>7}" for x in r[1:]))
for k, v in e["comparisons"].items():
    s = v.get("metrics", {}).get("success", v)
    print(k, "success delta", s.get("delta"), "p", s.get("p_value"))
for name, c in e["checkpoints"].items():
    if "summary" in c:
        print(name, {l: m["success"] for l, m in c["summary"]["per_language"].items()})

In [ ]:
subprocess.run(f"cd {REPO} && zip -qr /content/results.zip results", shell=True)
from google.colab import files; files.download("/content/results.zip")